# 14 — Run the whole pipeline for every grade in the dataset

**What this is.** One notebook that takes every design family in your dataset, rebuilds the
order the plant actually placed for it, and runs it through Layer 0 → screen → SERVE →
EXPLORE.

**Why it matters here.** Notebooks 11 and 12 demonstrate one hand-written M30 order. This
runs the orders your data actually records — five grades, each with its own exposure, use,
slump and materials.

**What breaks if it is ignored.** You only ever see one grade, and it was built from my
worked-example inputs (OPC), not from your plant (PPC).

---

## Where every input comes from

Nothing here is invented. Each value is either **read from the dataset** for that design
family, or is a **declared input** you must confirm. Every output row says which.

| input | source | classification |
|---|---|---|
| grade, exposure, RCC/PCC, target slump | the family's recorded values | ENGINEER INPUT (as recorded) |
| pumpable | majority `placing_method` of the family | ENGINEER INPUT (as recorded) |
| cement type | recorded — **PPC** | MATERIAL-SPECIFIC |
| cement / sand / coarse SG, admixture SG | family medians | MATERIAL-SPECIFIC |
| 600 µm passing → sand zone | family median → `[IS]` IS 383 Table 9 | MATERIAL-SPECIFIC → DERIVED FROM STANDARD |
| cement 28-day strength | family median of the certificates | MATERIAL-SPECIFIC |
| concrete temperature | family median | PLANT-SPECIFIC |
| SCMs available | **fly ash only** — GGBS is 0 in every row | PLANT-SPECIFIC |
| sand / coarse absorption, admixture solids, 20 : 10 mm split | family medians | MATERIAL-SPECIFIC |
| aggregate moisture for the batch sheet | family median — **measure on the day** | PLANT-SPECIFIC |
| IS 456 cl 5.1 suitability data (PPC + fly ash) | count of the plant's batches with 28-day cubes | EXPERIMENTALLY DETERMINED |
| **fly ash specific gravity** | **not in the dataset** | **declared below — confirm it** |
| **admixture dosage range** | **not in the dataset** | **declared below — confirm it** |

## Run order

`09 → 10 → 10b → 11 → 12 → 14`

This notebook imports the modules those notebooks export. **11 and 12 must have been run
first** so that `screen.py` and `selection.py` exist.

In [ ]:
# Always load the CURRENT project modules from disk. Python imports a module once per
# kernel, so a kernel that was open before an earlier notebook rewrote layer0.py /
# screen.py / ... would silently keep the OLD code. Dropping them first makes every
# Run All pick up the latest files, with or without a kernel restart.
import sys as _sys
for _m in ("is_rules", "layer0", "workability_residual", "strength_residual",
           "screen", "selection", "loop"):
    _sys.modules.pop(_m, None)

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import is_rules as IS
import layer0 as L0
import workability_residual as WR
import strength_residual as SR
import screen as SC
import selection as SEL

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

wmodel = joblib.load("workability_residual_model.joblib")
smodel = SC.load_strength_model()
assert smodel is not None, "run 10b_strength_residual_GP.ipynb first"
print("models loaded: workability", wmodel["n"], "rows / strength", smodel["n"], "samples")

## 1. The two inputs your dataset does not contain — confirm these

Everything else is read from your data. These two are not recorded anywhere in it, so they
are declared here, used for every grade, and **stamped on every output row**.

In [ ]:
# ---------------------------------------------------------------------------------
# CONFIRM BOTH VALUES BEFORE TRUSTING ANY OUTPUT
# ---------------------------------------------------------------------------------

# Fly ash specific gravity. IS 3812 (Part 1) specifies fineness and chemistry, not a
# design specific gravity; it comes from the Class F fly ash test certificate.
# Not present in the dataset. Replace with the certificate value.
# Searched 2026-10-01: no fly ash certificate or PCE datasheet for Plant D in any
# connected folder. Sensitivity check: fly ash SG 2.0 / 2.2 / 2.4 gives the same verdict
# and the same recommended mixes; see docs/missing-inputs-check.md.
FLYASH_SG = 2.20
FLYASH_SG_SOURCE = "ASSUMED -- not in the dataset; replace with the fly ash certificate value"

# Admixture dosage range, % of binder. IS 9103 prints no numeric cap; cl 10.2 requires
# the MANUFACTURER to state the recommended dosage. Not present in the dataset.
# Default below is the range this plant actually dosed (0.621-1.476 %), which keeps the
# search inside what the cl 5.3 water fit was built on. Replace with the PCE datasheet.
ADMIXTURE_RANGE_PCT = (0.621, 1.476)
ADMIXTURE_RANGE_SOURCE = ("PLANT-OBSERVED dose range, not the IS 9103 cl 10.2 manufacturer "
                          "declaration -- replace with the PCE datasheet range")

# [PC] SERVE feasibility thresholds -- the same ones notebook 12 uses
P_STRENGTH_MIN = 0.90
P_SLUMP_MIN = 0.80

# [PC] 5 designs, 3 IS 10262 cl 5.8 trial batches and 3 exploration proposals per grade
#      -- set in selection.py (notebook 12): N_DESIGNS, N_EXPLORE, N_CUBES.

print(f"fly ash SG      {FLYASH_SG}   <- {FLYASH_SG_SOURCE}")
print(f"admixture range {ADMIXTURE_RANGE_PCT} %   <- {ADMIXTURE_RANGE_SOURCE}")

## 2. Rebuild each recorded order from the dataset

`screen.plant_orders()` (notebook 11) builds one order and one `MaterialState` per design
family from the plant's own records. It also records the IS 456 cl 5.1 suitability
evidence for the plant's cement + fly ash combination.

In [ ]:
ORDERS = SC.plant_orders(flyash_sg=FLYASH_SG, admixture_range_pct=ADMIXTURE_RANGE_PCT)
s = pd.read_csv("03_strength_dataset.csv")
fc_by_fam = s.groupby("design_grp")["fc_mpa"].median()

for o in ORDERS:
    if "error" in o:
        print(f"{o['family']:16s} SKIPPED -- {o['error']}")
        continue
    r, st = o["req"], o["state"]
    o["plant"]["plant_fc_measured"] = float(fc_by_fam.get(o["family"], np.nan))
    print(f"{o['family']:16s} {r.grade}  {r.exposure.name:8s} {r.use.name}  slump {r.target_slump_mm:5.0f}"
          f"  pumped {str(o['pumped']):5s}  {st.cement_type}  600um {st.sand_passing_600um:5.1f}"
          f"  SCM {st.scms_available}  n={o['n_batches']}")
print("\ncl 5.1 evidence (first family):", ORDERS[0]["state"].combination_suitability_evidence)

## 3. Run every order — 5 designs, 3 IS trial batches, 3 exploration proposals each

For each grade: Layer 0 first (does IS 456 Table 5 allow this order at all?), then
`selection.lab_package()` — the same function notebook 12 uses for the M30 order.

In [ ]:
train = pd.read_csv("04_workability_dataset.csv")
train = train[train["design_grp"].astype(str).str.contains("[A-Za-z]", na=False)]
SUPPORT = SC.training_support(train, wmodel["features"])


def layer0_precheck(req, state):
    """Does IS 456 Table 5 allow this order at all, before any search?"""
    t5 = IS.table5_lookup(req.exposure, req.use, int(state.msa_mm))
    g = lambda x: int(str(x).upper().lstrip("M"))
    if t5["min_grade"] and g(req.grade) < g(t5["min_grade"]) and not t5.get("min_grade_disapplied_by"):
        return (f"REFUSED -- [IS] IS 456 Table 5: minimum grade for {req.exposure.name} "
                f"{req.use.name} is {t5['min_grade']}; this order is {req.grade}")
    return None


summary, designs_all, trials_all, batches_all, explore_all = [], [], [], [], []
for o in ORDERS:
    if "error" in o:
        continue
    req, st, fam = o["req"], o["state"], o["family"]
    row = {"family": fam, "grade": req.grade, "exposure": req.exposure.name,
           "use": req.use.name, "target_slump_mm": req.target_slump_mm,
           "pumped": o["pumped"], "n_batches": o["n_batches"], **o["plant"],
           "flyash_sg": FLYASH_SG, "flyash_sg_source": FLYASH_SG_SOURCE,
           "admixture_range_pct": str(ADMIXTURE_RANGE_PCT),
           "admixture_range_source": ADMIXTURE_RANGE_SOURCE,
           "cl_5_1_evidence": st.combination_suitability_evidence}
    refusal = layer0_precheck(req, st)
    if refusal:
        row.update({"status": "REFUSED AT LAYER 0", "reason": refusal})
        summary.append(row)
        print(f"\n{fam}: {refusal}")
        continue
    P = SEL.lab_package(req, st, wmodel=wmodel, smodel=smodel, support=SUPPORT,
                        order_id=fam, p_strength_min=P_STRENGTH_MIN, p_slump_min=P_SLUMP_MIN,
                        sand_moisture_pct=o["plant"]["sand_moisture_median"],
                        ca_moisture_pct=o["plant"]["ca_moisture_median"])
    row.update({"status": P["status"], "serve_candidates": P["n_serve"],
                "n_designs": len(P["designs"]), "n_trials": len(P["trials"]),
                "notes": " | ".join(P["notes"])})
    if len(P["designs"]):
        best = P["designs"].iloc[0]
        row.update({"rec_binder": best.binder_kg_m3, "rec_cement": best.cement_kg_m3,
                    "rec_flyash": best.flyash_kg_m3, "rec_water": best.water_kg_m3,
                    "rec_sp_pct": best.sp_pct_binder, "rec_wb": best.wb_ratio,
                    "rec_wc_counted": best.wc_counted, "rec_fc_pred": best.fc_pred,
                    "rec_target_mean": best.target_mean, "rec_P_strength": best.P_strength,
                    "rec_slump_pred": best.slump_pred, "rec_P_slump": best.P_slump,
                    "validation_level": "COMPUTATIONALLY_COMPLIANT"})
        designs_all.append(P["designs"].assign(family=fam, grade=req.grade))
    else:
        row["best_P_strength"] = P["best_P_strength"]
    for name, lst in (("trials", trials_all), ("batches", batches_all), ("explore", explore_all)):
        if len(P[name]):
            lst.append(P[name].assign(family=fam, grade=req.grade))
    summary.append(row)
    print(f"\n{fam}: {P['status']}   designs {len(P['designs'])}   trials {len(P['trials'])}"
          f"   explore {len(P['explore'])}")
    for n in P["notes"]:
        print("   - " + n)

SUMMARY = pd.DataFrame(summary)

## 4. Results

In [ ]:
cols = ["grade", "exposure", "target_slump_mm", "status", "serve_candidates", "n_designs",
        "n_trials", "rec_binder", "rec_cement", "rec_flyash", "rec_water", "rec_sp_pct",
        "rec_wc_counted", "rec_fc_pred", "rec_target_mean", "rec_P_strength", "rec_P_slump"]
show = SUMMARY[[c for c in cols if c in SUMMARY]]
print(show.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))

if trials_all:
    T = pd.concat(trials_all, ignore_index=True)
    print("\nIS 10262 cl 5.8 trial batches (same water, w/cm about +/-10 %):")
    print(T[["grade", "trial_no", "binder_kg_m3", "cement_kg_m3", "flyash_kg_m3", "water_kg_m3",
             "sp_pct_binder", "wb_ratio", "wc_counted", "fc_pred", "target_mean", "slump_pred"]]
          .to_string(index=False, float_format=lambda x: f"{x:8.3f}"))

print("\nRecommended mix vs what the plant actually batched:")
for _, r in SUMMARY.iterrows():
    if pd.isna(r.get("rec_binder")):
        why = r.get("reason") if isinstance(r.get("reason"), str) else r.get("notes", "")
        bp = r.get("best_P_strength")
        print(f"  {r['grade']}: {r.get('status')} -- {why}"
              + (f" (best P_strength {bp:.2f})" if isinstance(bp, float) and bp == bp else ""))
        continue
    print(f"  {r['grade']}: binder {r.rec_binder:.0f} (plant {r.plant_binder:.0f})"
          f"   fly ash {r.rec_flyash:.1f} (plant {r.plant_flyash:.1f})"
          f"   water {r.rec_water:.1f} (plant {r.plant_water:.1f})"
          f"   w/b {r.rec_wb:.3f} (plant {r.plant_wb:.3f})"
          f"   fc {r.rec_fc_pred:.1f} predicted vs {r.plant_fc_measured:.1f} measured")

In [ ]:
def _write(lst, path, what):
    if lst:
        df = pd.concat(lst, ignore_index=True)
        df.to_csv(path, index=False)
        print(f"wrote {path:36s} -- {len(df)} {what}")
    else:
        Path(path).unlink(missing_ok=True)
        print(f"no {what}; {path} not written")

SUMMARY.to_csv("14_all_grades_summary.csv", index=False)
print("wrote 14_all_grades_summary.csv")
_write(designs_all, "14_all_grades_designs.csv", "designs (up to 5 per grade, SERVE)")
_write(trials_all, "14_all_grades_trials.csv", "IS 10262 cl 5.8 trial batches (3 per grade)")
_write(batches_all, "14_all_grades_trial_batches.csv", "batch-sheet rows")
_write(explore_all, "14_all_grades_explore_proposals.csv", "optional exploration proposals")
for old in ("14_all_grades_explore_trials.csv", "14_all_grades_serve_front.csv"):
    if Path(old).exists():
        print(f"note: {old} is from an earlier version and is superseded")
print()
print("Every mix is COMPUTATIONALLY COMPLIANT and nothing more. None has been cast.")
print(f"Every row carries: fly ash SG {FLYASH_SG} [{FLYASH_SG_SOURCE.split(' --')[0]}],")
print(f"admixture range {ADMIXTURE_RANGE_PCT} [{ADMIXTURE_RANGE_SOURCE.split(' dose')[0]}].")
print("Cost and carbon are PLACEHOLDER rates.")

## Reading the results honestly

- **Validation level.** Every recommended mix is `COMPUTATIONALLY_COMPLIANT`. None has
  been cast. The industry-expert review and a physical trial come next — notebook 13.
- **Slump.** The slump model fails two of its own gates (G6, G9 — see `docs/audit-final.md`).
  The slump side of any recommendation is a proposal to confirm by trial.
- **Strength.** The strength model passes all its gates. Fig 1 is applied with the OPC 43
  curve because `[IS]` IS 10262 Annex A-5 directs PPC to that curve.
- **EXPLORE.** `fc_pred` and `slump_pred` on an EXPLORE row outside training support are not
  predictions — the model has not seen anything like it. Those rows are things to test.
- **A refusal at Layer 0 is an output.** If a grade's recorded order is below the IS 456
  Table 5 minimum for its exposure, the system refuses it and says which clause — it does
  not search around the standard.